<a href="https://colab.research.google.com/github/prometheus404/NLP_proj/blob/master/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NLP project

In [7]:
%pip install llama-cpp-python==0.2.90 --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122

Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cu122
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 443.8/443.8 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.2 MB/s eta 0:00:00


In [8]:
from llama_cpp import Llama, llama_free, llama_free_model
from tqdm import tqdm
#from transformers import AutoTokenizer, pipeline, BitsAndBytesConfig
import requests
from collections import defaultdict
import json
import torch


In [9]:
# Load the model
VERBOSE = True
CHOSEN = 'llama'
models = {
    'llama': {'repo_id':"bartowski/Meta-Llama-3.1-8B-Instruct-GGUF",
              'filename':"Meta-Llama-3.1-8B-Instruct-Q8_0.gguf"},
    'qwen': {'repo_id':"Qwen/Qwen3-8B-GGUF",
             'filename': "Qwen3-8B-Q8_0.gguf"},
    'mistral': {'repo_id':"TheBloke/Mistral-7B-v0.1-GGUF",
                'filename':"mistral-7b-v0.1.Q8_0.gguf"},
}

model = Llama.from_pretrained(repo_id=models[CHOSEN]['repo_id'], # repository name
                            filename=models[CHOSEN]['filename'], # model file
                            n_gpu_layers=-1, # use all GPU layers
                            n_ctx=32768, # context size
                            flash_attn=True, # use flash attention
                            chat_format="llama-3", # chat format
                            verbose=False)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


./Meta-Llama-3.1-8B-Instruct-Q8_0.gguf:   0%|          | 0.00/8.54G [00:00<?, ?B/s]

In [10]:
rulebook = requests.get('https://raw.githubusercontent.com/prometheus404/NLP_proj/refs/heads/master/rules/texts/dominion.txt').text
rulebook[:100]

'# Dominion\nYou are a monarch, like your parents before you - a ruler of a small pleasant kingdom of '

In [11]:
# Check that input is inside context window
#tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.1")

#tokens = tokenizer.encode(rulebook)
#print(len(tokens))

In [12]:
def generate_message(prompt, rulebook):
    return [
        {
                "role": "system",
                "content": prompt,
            },
            {
                "role": "user",
                "content": "Here is the rulebook:\n"+rulebook,
            },
    ]

In [13]:
import gc
def multiple_model_test(prompts, file_names, iterations, test_name):
    outputs = defaultdict(dict)

    for game,prompt_name,prompt,it in tqdm([(f,pn,p,it) for f in file_names for (pn,p) in prompts for it in range(iterations)]):
        rulebook = requests.get('https://raw.githubusercontent.com/prometheus404/NLP_proj/refs/heads/master/rules/texts/'+game+'.txt').text
        out = model.create_chat_completion(generate_message(prompt, rulebook), temperature=0.7)
        outputs[CHOSEN+'-'+game+'-'+prompt_name][str(it)] = out['choices'][0]['message']['content']

    with open(f'{test_name}.out','w') as f:
        json.dump(dict(outputs),f)

    return outputs

# Rule extraction
1. Give the model a rulebook and prompt it to explain the game in simple, conversational terms to a child or other audiences. -> tree decomposition to test how well the model did
2. Test the ability of the model to find analogies of rules (?)
3. Test the ability to extract if-then rules (?)
4. Organize the rules of into a hierarchy: top-level objectives, mid-level phases, low-level actions. (?) (look into the paper)

In [14]:
prompts = [("kid", """You are a friendly tutor explaining board games to a 7‑year‑old. Summarize the game in plain language, using short sentences and with fun tone. Include:
                - Goal of the game
                - How a player wins
                - What a turn looks like
                - Exceptions to standard rules

                The user will give you a text file with the rulebook you need to explain.
                the output should not be too long. All rules must be present in your explanation"""),
           ("analogies", """The user will give you a text file with the rulebook you need to explain
           the output should not be too long. All rules must be present in your explanation.
           Explain the rules to a child by comparing it to something they already know (e.g. some other famous board games).
           se the rulebook to keep the analogy accurate, and end with a one‑sentence “what you try to achieve” statement.\n"""),
]
game_names = [ 'dominion','7_wonders', 'catan', 'power_grid_recharged','ticket_to_ride',]

iterations = 1

multiple_model_test(prompts, game_names, iterations, 'extraction')

100%|██████████| 10/10 [03:49<00:00, 22.93s/it]


defaultdict(dict,
            {'llama-dominion-kid': {'0': "Let's learn how to play Dominion!\n\n**Goal of the game:**\nYour goal is to build the best deck of cards by collecting the most valuable cards. The player with the most points (called <shield>) at the end of the game wins!\n\n**How a player wins:**\nThe game ends when three or more Supply piles are empty, or the Province pile is empty. Then, players count up their points (shield). The player with the most points wins!\n\n**What a turn looks like:**\nA turn has three phases: Action, Buy, and Clean-up.\n\n1. **Action phase:** You can play one Action card from your hand. Action cards are special cards that let you do something cool, like gain a card, draw a card, or trash a card.\n2. **Buy phase:** You can play Treasure cards from your hand to earn coins. Then, you can use those coins to buy one card from the Supply.\n3. **Clean-up phase:** You discard all the cards you played and hand cards, and draw a new hand of five cards.\n\

## Error detection

 Each rulebook is edited by inserting a set of 5 errors each of increasing difficulty:
 - level 0: **original** -> unaltered rulebook
 - level 1: **missing** -> an entire paragraph of the rulebook describing some core mechanic is missing
- level 2: **unsolvable** -> (in one line states you can draw two cards, in another one that you can draw only one)
- level 3: **incoherent** -> a mechanic that hardlocks the game (you cannot play train if you do not have train on the map but on another line clearly states you start with an empty map)
- level 4: **gamebreaking** -> a coherent but obviously unbalanced mechanic

In [22]:
prompts = ["""You are an expert game board player.
Examine the rulebook provided by the user.
Proceed with a chain of thought:

- Scan the text linearly and note any statements that conflict with earlier ones.
- Look for gaps where a required mechanic is not explained.
- Check whether any mechanic could halt the game or give a player an overwhelming advantage.

Report the **most gamebreaking** problem you discover, quoting the relevant line and summarizing its impact.
If nothing stands out, reply “The rules appear consistent.”
"""]
file_names = ['ticket_to_ride', 'dominion', 'catan', 'power_grid_recharged']
iterations = 5


output_dict = {str(g):{'lvl'+str(l): {str(it): '' for it in range(iterations)} for l in range(5)} for g  in game_names}


for game,prompt,it,lvl in tqdm([(f,p,it,lvl) for f in file_names for p in prompts for it in range(iterations) for lvl in range(5)]):
    rulebook = requests.get('https://raw.githubusercontent.com/prometheus404/NLP_proj/refs/heads/master/rules/flawed_texts/lvl'+str(lvl)+'/'+game+'.txt').text
    out = model.create_chat_completion(generate_message(prompt, rulebook), temperature=0.7)['choices'][0]['message']['content']
    if(VERBOSE):
        print(out)
    output_dict[game]['lvl'+str(lvl)][str(it)] = out

with open(f'{CHOSEN}_error_detection.out','w') as f:
    json.dump(dict(output_dict),f)

  1%|          | 1/100 [00:12<21:20, 12.93s/it]

After scanning the rulebook, I found a potential gamebreaking problem.

**Most gamebreaking problem:** The rule for drawing train cards has a critical inconsistency.

**Quoted line:** "If, at any time, 3 of the 5 face up train cards are locomotives, all 5 cards are immediately discarded, and 5 new cards are turned face up to replace them."

**Impact:** This rule allows a player to deliberately draw locomotive cards to trigger the discard of the entire face-up deck, effectively removing all possible options for drawing train cards for the next player. This can lead to a player having a significant advantage, as they can control the availability of train cards for their opponents.

In particular, if a player has a locomotive card and chooses to draw a face-up locomotive, they can immediately discard the entire deck, forcing the next player to draw from the bottom of the deck or take the "claim 1 route" or "draw tickets" actions. This can create a situation where the player who discarded 

  1%|          | 1/100 [00:25<42:10, 25.56s/it]

After examining the rulebook, I've identified a potential game-breaking problem.

**Most game-breaking problem:**

"The game can end prematurely when a player runs out of trains, but the game doesn't specify what happens to the remaining train cards and tickets when the game ends."

**Relevant line:**

"When a player’s stock of plastic trains gets down to only 0,1, or 2 trains left at the end of their turn, each player, including that player, gets 1 final turn."

**Impact:**

This problem can lead to a situation where the game ends early, and the remaining train cards and tickets are not resolved. For example, if a player has a long path that can be completed with the remaining trains, but the game ends before they can claim those routes, they may lose points that they could have earned. Similarly, if a player has a ticket that can be completed with the remaining trains, but the game ends before they can complete it, they may lose points for that ticket.

To mitigate this issue, the ga

KeyError: 'lvl1'

In [ ]:
# EVALUATION
sys_prompt = """You are a rule‑checking assistant.
Read ANSWER and GROUND TRUTH, narrate whether the explanations refer to the same rule, and conclude with a single word **EQUAL** or **DIFFERENT** on the final line.

### Examples
input:'''
ANSWER -> The Monopoly win condition states that the game ends when all other players have gone bankrupt,leaving a single player with all the assets. However, this rule can be problematic because it assumes that players will continue to take turns until bankruptcy occurs, which may never happen if a player repeatedly lands on “Free Parking” and collects cash without paying any fees. In such a scenario, the game could drag on indefinitely, making the win condition effectively unreachable. A more practical rule might impose a turn limit or a cash‑threshold to declare a winner when the game stalls.
GROUND TRUTH-> The problem is that the first player may reroll the dice any time.
'''

output:'''
The answer does not talk about rerolling the dice
DIFFERENT
'''

---

input:'''
ANSWER ->  After examining the rulebook, I've identified a **gamebreaking** problem: In the first phase of the game rules states that when a player draws a card any other player can choose to draw another card. This could create a scenario where all player are willing to draw until the deck is empty.
GROUND TRUTH -> infinite card draw at the start of the game.
Both the answer and the ground truth adress the card draw in first phases of the game as the problem
EQUAL
"""
ground_truth = {
    "ticket_to_ride": {
        "lvl0": "the rulebook is coherent",
        "lvl1": "delete the claim route section",
        "lvl2": "draw 2 cards but on another line states draw up to 7 cards",
        "lvl3": "added \"a player cannot claim a route unless they have already claimed at least one other contiguous route earlier in the game\"",
        "lvl4": "Instead of drawing any of the two cards you can instead reveal cards from the top of the train deck until you reveal a locomotive card. Add the locomotive card to your hand and shuffle the revealed cards with the rest of the train deck."
    },
    "dominion": {
        "lvl0": "the rulebook is coherent",
        "lvl1": "remove buy phase section",
        "lvl2": "added: \"After buying a card, a player may **continue to play additional Treasure cards** from their hand to increase their total  for that turn\"",
        "lvl3": "added: \"You cannot play cards that grant more action as the first action of the turn\"",
        "lvl5": "can play treasures also from the discard pile"
    },
    "7_wonders": {
        "lvl0": "the rulebook is coherent",
        "lvl1": "remove resolution of military conflicts",
        "lvl2": "added \"When buying a resource you may use Coins received from a neighbor earlier in the same turn to pay the 2‑Coin cost.\"",
        "lvl3": "infinite resolve conflict loop",
        "lvl5": "When you construct the Building on your card, you can construct one stage of your Wonder for free. The effect of this stage is now available to you for the rest of the game."
    },
    "catan": {
        "lvl0": "the rulebook is coherent",
        "lvl1": "remove build subsection",
        "lvl2": "added \"When you roll a 7, all hexes produce a resource except for the one with the robber.\"",
        "lvl3": "You can only trade a resource produced this turn by the hex with the robber.",
        "lvl5": "robber steals all resources from all players"
    },
    "power_grid_recharged": {
        "lvl0": "the rulebook is coherent",
        "lvl1": "phase 5 now consist only of the first step (supply energy and generate money)",
        "lvl2": "modified phrase \"As explained in the section “The Power Plants”, each power plant may store only the number of resource tokens matching the number of symbols on the card and needs twice that number of tokens to produce electricity\"",
        "lvl3": "added 'Remove the “Step 3” card and place it in the game box' (no way to reach phase 3)",
        "lvl4": "A player who does not supply any city receives the same amount of Elektro as the highest‑earning player"
    }
}

verbose = True
res = []


for lvl in range(5):
    # construct the answer pair

    answer_pair = []
    for game in game_names:
        for x in output_dict[game][f'lvl{lvl}']:
            answer_pair.append((x, ground_truth[game][f'lvl{lvl}']))
    # check
    for answer, ground_truth in answer_pair:
        usr_prompt = f"ANSWER -> {answer}\nGROUND TRUTH -> {ground_truth}"
        out = model.create_chat_completion(generate_message(sys_prompt, usr_prompt), temperature=0.7)
        if(verbose):
            print(out['choices'][0]['message']['content'])
        res.append(out['choices'][0]['message']['content'].splitlines()[-1])
    print(f"lvl{lvl}: {res.count('EQUAL')}/{len(res)}")


# Game classification
Give the model a rulebook and ask it to classify the mechanics, evaluate the complexity, suggests the perfect number of players and estimate the duration

In [ ]:
prompts = ["""test. For"""]
file_names = ['ticket_to_ride', 'dominion', 'catan', 'power_grid_recharged']
iterations = 1

multiple_model_test(prompts, file_names, iterations, 'classification')


In [ ]:
# EVALUATION
